# 04. Model pricing with GARCH volatility

Цель этого блока - построить минимальный MVP benchmark, в котором волатильность для `Black-76` задаётся не историческим rolling-window, а через простой `GARCH(1,1)` на доходностях базового актива. Это позволяет проверить, даёт ли более адаптивный volatility input улучшение по сравнению с простым historical volatility baseline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from arch import arch_model

In [ ]:
project_root = Path('/Users/maria/Desktop/Code/HSE/COURSEBOOK')
input_path = project_root / 'data/final/imoex_mm_options_modelling_sample.parquet'
results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(input_path)
df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'], errors='coerce')
df['market_price'] = pd.to_numeric(df['SETTLEPRICE'], errors='coerce')
df['F'] = pd.to_numeric(df['UNDERLYING_CLOSE'], errors='coerce')
df['K'] = pd.to_numeric(df['strike'], errors='coerce')
df['T'] = pd.to_numeric(df['DTE_DAYS'], errors='coerce') / 365.0
df['r'] = 0.16
df['option_type_norm'] = df['option_type'].astype(str).str.upper().map({'C': 'call', 'P': 'put', 'CALL': 'call', 'PUT': 'put'})
df.shape

## 1. GARCH(1,1) on underlying returns

Берём дневные лог-доходности `UNDERLYING_CLOSE`, строим на них простой `GARCH(1,1)` и используем условную волатильность как основу для прогноза на следующий день. Для MVP это достаточно: цель здесь не идеальная спецификация модели волатильности, а рабочий benchmark для сравнения качества ценообразования.

In [ ]:
underlying = (
    df[['TRADEDATE', 'UNDERLYING_CLOSE']]
    .drop_duplicates()
    .sort_values('TRADEDATE')
    .reset_index(drop=True)
)
underlying['ret_1d'] = np.log(underlying['UNDERLYING_CLOSE']).diff()

returns_pct = underlying['ret_1d'].dropna() * 100.0
garch = arch_model(returns_pct, mean='Zero', vol='GARCH', p=1, q=1, dist='normal', rescale=False)
garch_fit = garch.fit(disp='off', update_freq=0)

underlying.loc[underlying['ret_1d'].notna(), 'garch_daily_vol'] = garch_fit.conditional_volatility.values / 100.0
underlying['garch_daily_vol_forecast'] = underlying['garch_daily_vol'].shift(1)
underlying['garch_vol'] = underlying['garch_daily_vol_forecast'] * np.sqrt(252)

print(garch_fit.params)
underlying[['TRADEDATE', 'UNDERLYING_CLOSE', 'ret_1d', 'garch_vol']].head()

## 2. Merge GARCH volatility back to options

Теперь присоединяем годовую `garch_vol` обратно к опционным наблюдениям по `TRADEDATE`. После этого у нас есть полный набор входов для `Black-76`.

In [ ]:
df = df.merge(underlying[['TRADEDATE', 'garch_vol']], on='TRADEDATE', how='left')

df['DTE_bucket_garch'] = pd.cut(
    df['DTE_DAYS'],
    bins=[0, 7, 30, 90, np.inf],
    labels=['0-7', '8-30', '31-90', '91+'],
)

df['moneyness_bucket_garch'] = pd.cut(
    df['moneyness'],
    bins=[0, 0.9, 0.97, 1.03, 1.1, np.inf],
    labels=['deep_OTM', 'OTM', 'ATM', 'ITM', 'deep_ITM'],
)

float(df['garch_vol'].notna().mean())

## 3. Black-76 pricing with GARCH volatility

Используем `garch_vol` как входную волатильность и считаем модельную цену. Здесь важно фильтровать только очевидно невалидные строки: нулевые цены, нулевой `F/K/T`, отсутствие `garch_vol`, некорректный тип опциона.

In [ ]:
def black76_price(F, K, T, r, sigma, option_type):
    F = np.asarray(F, dtype=float)
    K = np.asarray(K, dtype=float)
    T = np.asarray(T, dtype=float)
    r = np.asarray(r, dtype=float)
    sigma = np.asarray(sigma, dtype=float)

    sqrt_T = np.sqrt(T)
    d1 = (np.log(F / K) + 0.5 * sigma**2 * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    disc = np.exp(-r * T)

    call_price = disc * (F * norm.cdf(d1) - K * norm.cdf(d2))
    put_price = disc * (K * norm.cdf(-d2) - F * norm.cdf(-d1))
    return np.where(np.asarray(option_type) == 'call', call_price, put_price)


valid_mask = (
    (df['market_price'] > 0)
    & (df['F'] > 0)
    & (df['K'] > 0)
    & (df['T'] > 0)
    & (df['garch_vol'] > 0)
    & (df['option_type_norm'].isin(['call', 'put']))
)

df['model_price_garch'] = np.nan
df.loc[valid_mask, 'model_price_garch'] = black76_price(
    F=df.loc[valid_mask, 'F'],
    K=df.loc[valid_mask, 'K'],
    T=df.loc[valid_mask, 'T'],
    r=df.loc[valid_mask, 'r'],
    sigma=df.loc[valid_mask, 'garch_vol'],
    option_type=df.loc[valid_mask, 'option_type_norm'],
)

df['error_garch'] = df['model_price_garch'] - df['market_price']
df['abs_error_garch'] = df['error_garch'].abs()
df['pct_error_garch'] = df['error_garch'] / df['market_price']
df['squared_error_garch'] = df['error_garch'] ** 2

df[['market_price', 'garch_vol', 'model_price_garch', 'error_garch']].head()

## 4. Overall metrics

Сначала посмотрим на качество модели на всей выборке. Это даёт главный ответ: насколько вообще рабочим оказался `Black-76 + GARCH volatility` как baseline.

In [ ]:
def calc_metrics(frame):
    sub = frame.loc[frame['model_price_garch'].notna()].copy()
    return pd.Series(
        {
            'N': int(len(sub)),
            'MAE': float(sub['abs_error_garch'].mean()),
            'RMSE': float(np.sqrt(sub['squared_error_garch'].mean())),
            'mean_error': float(sub['error_garch'].mean()),
            'median_abs_error': float(sub['abs_error_garch'].median()),
        }
    )


overall_metrics = calc_metrics(df).to_frame().T
overall_metrics

**Короткая интерпретация.** Если сравнивать с historical-vol baseline, GARCH можно оценивать по тем же метрикам `MAE` и `RMSE`. Если ошибки не уменьшаются, это значит, что более сложный volatility block сам по себе ещё не гарантирует лучшего pricing.

## 5. Metrics by regime and buckets

Дальше разложим ошибки по трём важным разрезам:
- `vol_regime`;
- `DTE bucket`;
- `moneyness bucket`.

Это поможет понять, где именно GARCH-baseline работает лучше или хуже.

In [ ]:
def summarize_by_group(frame, group_col):
    sub = frame.loc[frame['model_price_garch'].notna()].copy()
    out = (
        sub.groupby(group_col, dropna=False, observed=False)
        .agg(
            N=('error_garch', 'size'),
            MAE=('abs_error_garch', 'mean'),
            mean_error=('error_garch', 'mean'),
            median_abs_error=('abs_error_garch', 'median'),
        )
        .reset_index()
    )
    rmse = (
        sub.groupby(group_col, dropna=False, observed=False)['squared_error_garch']
        .apply(lambda s: float(np.sqrt(np.mean(s))))
        .reset_index(name='RMSE')
    )
    return out.merge(rmse, on=group_col, how='left')


vol_regime_metrics = summarize_by_group(df, 'vol_regime')
dte_metrics = summarize_by_group(df, 'DTE_bucket_garch')
moneyness_metrics = summarize_by_group(df, 'moneyness_bucket_garch')

print('By vol_regime')
display(vol_regime_metrics)
print('\nBy DTE bucket')
display(dte_metrics)
print('\nBy moneyness bucket')
display(moneyness_metrics)

**Короткая интерпретация.** Для GARCH-baseline обычно стоит ожидать той же общей структуры, что и у historical-vol benchmark: ошибки должны быть больше в `high_vol`, на длинных `DTE` и на хвостах `moneyness`. Если этого не происходит, это означает, что GARCH меняет не только уровень ошибки, но и сам профиль модельных промахов.

## 6. Simple plots

Ниже - три простых графика, которых достаточно для MVP-анализа: по regime, по сроку и по динамике ошибок во времени.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

vol_plot = vol_regime_metrics.dropna(subset=['vol_regime']).copy()
axes[0].bar(vol_plot['vol_regime'].astype(str), vol_plot['MAE'])
axes[0].set_title('MAE by vol_regime')
axes[0].set_ylabel('MAE')

dte_plot = dte_metrics.dropna(subset=['DTE_bucket_garch']).copy()
axes[1].bar(dte_plot['DTE_bucket_garch'].astype(str), dte_plot['MAE'])
axes[1].set_title('MAE by DTE bucket')
axes[1].set_ylabel('MAE')

rolling = (
    df.loc[df['model_price_garch'].notna()]
    .groupby('TRADEDATE')['abs_error_garch']
    .mean()
    .rolling(21)
    .mean()
)
axes[2].plot(rolling.index, rolling.values)
axes[2].set_title('Rolling 21-day mean abs error')
axes[2].set_ylabel('MAE')

plt.tight_layout()
plt.show()

## 7. Final short conclusion

- `GARCH(1,1)` даёт более структурированный volatility input, чем простой rolling-window.
- Но итоговую пользу надо оценивать только через pricing errors: если `MAE/RMSE` не снижаются, усложнение модели волатильности не окупается.
- Даже если overall improvement небольшой, GARCH может быть полезен тем, что по-другому ведёт себя в stress regimes.
- Следующий естественный шаг после этого блока - прямое сравнение `historical volatility` vs `GARCH volatility` в одной таблице и, возможно, добавление implied-vol based benchmark.

## 8. Save outputs

In [ ]:
result_path = results_dir / 'model_pricing_garch.parquet'
metrics_path = results_dir / 'model_pricing_garch_metrics.csv'

metrics_tables = []
metrics_tables.append(overall_metrics.assign(slice_type='overall', slice_value='all'))
metrics_tables.append(vol_regime_metrics.rename(columns={'vol_regime': 'slice_value'}).assign(slice_type='vol_regime'))
metrics_tables.append(dte_metrics.rename(columns={'DTE_bucket_garch': 'slice_value'}).assign(slice_type='DTE_bucket'))
metrics_tables.append(moneyness_metrics.rename(columns={'moneyness_bucket_garch': 'slice_value'}).assign(slice_type='moneyness_bucket'))
metrics_df = pd.concat(metrics_tables, ignore_index=True, sort=False)

df.to_parquet(result_path, index=False)
metrics_df.to_csv(metrics_path, index=False)

print('Saved:', result_path)
print('Saved:', metrics_path)